# State-dependent modulation of spiny projection neurons controls levodopa-induced dyskinesia in a mouse model of Parkinson's disease

This notebook shows how to find and read the files of DANDI:001538, the data of Zhai et al. 2025 from the Surmeier lab. The lab studied levodopa-induced dyskinesia (LID) in mice with a unilateral 6-OHDA lesion, a model of Parkinson's disease. The dataset holds:

- Patch-clamp recordings from striatal spiny projection neurons (dSPNs and iSPNs), with two-photon line scans of their dendrites
- Sr2+-oEPSC recordings: voltage-clamp currents evoked by optogenetic stimulation of cortical terminals
- GRAB-ACh3.0 two-photon imaging of acetylcholine release
- Abnormal involuntary movement (AIM) ratings and videos of the mice after levodopa
- Two-photon and confocal images of dendritic spines

## Streaming files from DANDI

The files can be read directly from the archive without downloading them. We list the dandiset's NWB files and define a function that opens one by its path.

In [ ]:
import h5py
import remfile
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

dandiset = DandiAPIClient().get_dandiset("001538", "draft")
assets = {asset.path: asset for asset in dandiset.get_assets() if asset.path.endswith(".nwb")}
print(f"{len(assets)} NWB files")


def open_nwbfile(asset_path):
    """Stream one NWB file from DANDI without downloading it."""
    url = assets[asset_path].get_content_url(follow_redirects=1, strip_query=True)
    io = NWBHDF5IO(file=h5py.File(remfile.File(url), "r"), load_namespaces=True)
    return io.read()

## Finding files

Each file's path on DANDI has the form

`sub-<subject_id>/sub-<subject_id>_ses-<session_id>_<modalities>.nwb`

The session ID has five parts joined by `++`:

`<cell type>++<state>++<pharmacology>++<genotype>++<date>`

For example, `dSPN++OnState++D1RaSch++WT++20190411` is a recording of dSPNs from a wild-type mouse in the LID on-state, with the D1 receptor antagonist SCH 23390, on 2019-04-11. Recordings that do not target one cell type (behaviour, acetylcholine imaging) have `pan` as the cell type.

The modalities suffix lists the kinds of data in the file, for example `icephys` for patch clamp and `ophys` for imaging. The helpers below read these parts from a path.

In [ ]:
from collections import Counter


def get_session_id(asset_path):
    """The session ID of a DANDI asset path, between '_ses-' and the modalities suffix."""
    file_name = asset_path.split("/")[-1].removesuffix(".nwb")
    return file_name.split("_ses-")[1].split("_")[0]


def get_modalities(asset_path):
    """The modalities suffix of a DANDI asset path, empty when the file has none."""
    after_session = asset_path.split("/")[-1].removesuffix(".nwb").split("_ses-")[1]
    return after_session.split("_")[1] if "_" in after_session else ""


def parse_session_id(session_id):
    """The five parts of a session ID."""
    cell_type, state, pharmacology, genotype, date = session_id.split("++")
    return {"cell_type": cell_type, "state": state, "pharmacology": pharmacology, "genotype": genotype, "date": date}


Counter(get_modalities(asset_path) or "(none)" for asset_path in assets)

## Patch-clamp recordings: one file per mouse and recording day

The patch-clamp files hold every cell the lab recorded from one mouse on one day. For each cell they have:

- One `CurrentClampSeries` with all its somatic sweeps stored one after another, each at its own start time, and one with its dendritic sweeps when the lab also recorded back-propagating spikes in the dendrites. The stimulus of each sweep is in a matching `CurrentClampStimulusSeries`.
- The intracellular recordings table and the tables above it (simultaneous recordings, sequential recordings, repetitions, experimental conditions), which group the sweeps by cell, protocol and condition. Each row of the intracellular recordings table points to its sweep's slice of the series.
- For dendritic sweeps, two-photon line scans across the dendrite in two dyes: Alexa Fluor 568, which fills the cell and shows its shape, and Fluo-4, which reports calcium.
- A `RecordingsIndexTable` in `processing/icephys` with one row per sweep that gathers the sweep's details, its rows in the tables above and its line-scan slices.

The Sr2+-oEPSC recordings are also patch-clamp files, with voltage-clamp series; they are described in their own section below.

In [ ]:
patch_clamp_paths = [asset_path for asset_path in assets if get_modalities(asset_path).startswith("icephys")]
print(f"Patch-clamp files: {len(patch_clamp_paths)}")

cohorts = Counter()
for asset_path in patch_clamp_paths:
    parts = parse_session_id(get_session_id(asset_path))
    cohorts[(parts["cell_type"], parts["state"], parts["pharmacology"], parts["genotype"])] += 1
for (cell_type, state, pharmacology, genotype), count in sorted(cohorts.items()):
    print(f"  {cell_type:5s} {state:18s} {pharmacology:9s} {genotype:10s} {count} files")

### Stream one file

We open a Figure 1 file: dSPNs of a mouse in the LID on-state, recorded before and after the D1 receptor antagonist SCH 23390 was added to the bath, with line scans of their dendrites.

In [ ]:
asset_path = (
    "sub-Subject20190411LIDONtDTomato/"
    "sub-Subject20190411LIDONtDTomato_ses-dSPN++OnState++D1RaSch++WT++20190411_icephys+ophys.nwb"
)
nwbfile = open_nwbfile(asset_path)

print(f"Subject: {nwbfile.subject.subject_id}")
print(f"Session start: {nwbfile.session_start_time}")
print(f"Cells (one electrode each): {len(nwbfile.icephys_electrodes)}")
print("Acquisition:")
for name in nwbfile.acquisition:
    print(f"  {name}")

### The RecordingsIndexTable

The quickest way into a file is the `RecordingsIndexTable`: one row per sweep, with the cell, the condition (named as in the paper's figure legends), the recording site (`Soma`, or a dendritic site such as `Proximal2`), the injected current and the sweep's start time.

In [ ]:
index_df = nwbfile.processing["icephys"]["RecordingsIndexTable"].to_dataframe()
print(f"RecordingsIndexTable: {len(index_df)} rows")
print(list(index_df.columns))
index_df[["cell_id", "condition", "site", "stimulus_current_pA", "sweep_start_time_s"]].head()

In [ ]:
# Sweeps per cell, condition and site
index_df.groupby(["cell_id", "condition", "site"]).size()

### Reading a sweep

Each row of the intracellular recordings table points to its sweep's slice of the cell's series: `idx_start` and `count` on the series in the `responses` column, and the same on the stimulus series in the `stimuli` column. A stimulus spans its command, so its slice can be shorter or longer than the recording. Here we plot the largest current step of cell 1 in each condition.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

recordings_df = nwbfile.intracellular_recordings.to_dataframe()
details = recordings_df["intracellular_recordings"]
somatic_df = recordings_df[(details["cell_id"] == 1) & (details["site"] == "Soma")]

fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True, sharey="row", height_ratios=[3, 1])
for column, (condition, condition_df) in enumerate(somatic_df.groupby(("intracellular_recordings", "condition"))):
    row = condition_df.loc[condition_df[("intracellular_recordings", "stimulus_current_pA")].idxmax()]
    response = row[("responses", "response")]
    stimulus = row[("stimuli", "stimulus")]
    time_s = response.timestamps - response.timestamps[0]
    axes[0, column].plot(time_s, response.data * response.timeseries.conversion * 1e3, lw=0.8)
    # The stimulus spans the command, which can end before the recording does
    stimulus_time_s = stimulus.timestamps - response.timestamps[0]
    axes[1, column].plot(stimulus_time_s, stimulus.data * stimulus.timeseries.conversion * 1e12, color="gray")
    axes[0, column].set_title(f"Cell 1, {condition}")
    axes[1, column].set_xlabel("Time from sweep start (s)")
axes[0, 0].set_ylabel("Membrane potential (mV)")
axes[1, 0].set_ylabel("Current (pA)")
plt.tight_layout()
plt.show()

print(f"Series: {response.timeseries.name}, {response.timeseries.data.shape[0]} samples for all its sweeps")
print(f"This sweep: samples {response.idx_start} to {response.idx_start + response.count}")

### The icephys tables

The tables above the intracellular recordings table group the sweeps. A cell recorded before and after a drug has one repetition per condition for the same protocol, and the experimental conditions table has one row per condition.

In [ ]:
repetitions_df = nwbfile.get_icephys_repetitions().to_dataframe()
repetitions_df[["cell_id", "protocol", "condition"]]

In [ ]:
nwbfile.icephys_experimental_conditions.to_dataframe()[["condition"]]

## Dendritic excitability: line scans

In the dendritic recordings the lab injected brief current pulses at the soma to evoke back-propagating spikes and imaged a line across a dendrite, close to the soma (proximal) or far from it (distal), at high speed. Each dendritic sweep has:

- Source images in `acquisition/ImageLineScanSource`: a snapshot of the field of view in each dye at each trial, for example `ImageAlexa568Cell1Proximal2Trial1`.
- Line scans in `acquisition`: the intensity along the line over time, one series per cell, site and dye with the site's trials in recording order, each at its own start time, for example `LineScanFluo4Cell1Proximal2`. The description gives the line's end points on the source images.
- ROI responses in `processing/ophys/Fluorescence`: the mean intensity over the pixels of the line that Prairie View averages into its line profile (the first 17, as a rule), for example `RoiResponseSeriesFluo4Cell1Proximal2`.
- Those pixels, the ROI, in `processing/ophys/ImageSegmentation`: one table per dye with one row per scan line, named in `roi_name` like the series, for example row `Cell1Proximal2` of `PlaneSegmentationFluo4`.

The `line_scan_alexa568`, `line_scan_fluo4`, `roi_response_alexa568` and `roi_response_fluo4` columns of the RecordingsIndexTable point to each sweep's slices.

In [ ]:
import matplotlib.colors as mcolors
from scipy.signal import butter, filtfilt

cell_id = 1
site = "Proximal2"
trial = 1  # the site's trials are numbered in recording order

site_df = index_df[(index_df["cell_id"] == cell_id) & (index_df["site"] == site)].sort_values("sweep_start_time_s")
trial_row = site_df.iloc[trial - 1]

source_image = nwbfile.acquisition["ImageLineScanSource"].images[f"ImageAlexa568Cell{cell_id}{site}Trial{trial}"]
source_array = np.asarray(source_image.data, dtype=float).T  # NWB images are indexed (x, y)

line_scan = trial_row["line_scan_fluo4"]
line_scan_data = np.asarray(line_scan.data, dtype=float)

roi_response = trial_row["roi_response_fluo4"]
roi_response_data = np.asarray(roi_response.data[:, 0], dtype=float)  # one column, for the one ROI
time_s = roi_response.timestamps - roi_response.timestamps[0]
sampling_rate = 1 / np.mean(np.diff(time_s))

# A first-order Butterworth low-pass at 10 Hz smooths the calcium signal
b, a = butter(1, 10.0 / (0.5 * sampling_rate), btype="low")
roi_response_filtered = filtfilt(b, a, roi_response_data)

# The pixels of the ROI on the source image, indexed (x, y) like the image
roi_x, roi_y = np.nonzero(roi_response.timeseries.rois[0]["image_mask"].iloc[0])

alexa568_cmap = mcolors.LinearSegmentedColormap.from_list("alexa568", ["black", "#FF4500"])
fluo4_cmap = mcolors.LinearSegmentedColormap.from_list("fluo4", ["white", "#32CD32"])

fig = plt.figure(figsize=(14, 8))
grid = fig.add_gridspec(nrows=2, ncols=2, hspace=0.3, wspace=0.25)

ax_source = fig.add_subplot(grid[:, 0])
ax_source.imshow(source_array, cmap=alexa568_cmap)
ax_source.plot(roi_x, roi_y, "yellow", linewidth=3, alpha=0.9, label="ROI")
ax_source.set_title(f"Alexa Fluor 568 source image, cell {cell_id} {site} trial {trial}")
ax_source.legend()

ax_line_scan = fig.add_subplot(grid[0, 1])
ax_line_scan.imshow(
    line_scan_data.T,
    cmap=fluo4_cmap,
    aspect="auto",
    origin="lower",
    extent=[0, time_s[-1], 0, line_scan_data.shape[1]],
)
ax_line_scan.set_title("Fluo-4 line scan")
ax_line_scan.set_xlabel("Time from sweep start (s)")
ax_line_scan.set_ylabel("Position along the line (pixels)")

ax_roi = fig.add_subplot(grid[1, 1])
ax_roi.plot(time_s, roi_response_data, color="#A5D6A7", linewidth=0.5, label="Raw")
ax_roi.plot(time_s, roi_response_filtered, color="#1B5E20", linewidth=1.5, label="Low-pass 10 Hz")
ax_roi.set_title("Fluo-4 mean over the ROI")
ax_roi.set_xlabel("Time from sweep start (s)")
ax_roi.set_ylabel("Fluorescence (a.u.)")
ax_roi.legend()
plt.show()

## Behaviour: AIM ratings and videos

The behaviour files hold one levodopa session of one mouse: the abnormal involuntary movement (AIM) ratings and the videos filmed to count contralateral rotations. Their session ID starts with `pan`, and their suffix is `image` when the session has videos.

- `nwbfile.events["aim_scores"]` has one row per rated category and observation: the time after the levodopa injection, the category (axial, limb, orolingual, or the lab's total) and the score.
- `processing/behavior/AimEthogram` is the catalogue of the rating scheme: what each category and score means, with HED tags.
- The videos are `ImageSeries` in `acquisition` whose frames stay in the video files next to the NWB file on DANDI.

In [ ]:
behavior_paths = sorted(
    asset_path
    for asset_path in assets
    if get_session_id(asset_path).startswith("pan++") and get_modalities(asset_path) in ("image", "")
)
print(f"Behaviour files: {len(behavior_paths)}")

# A session with videos
behavior_nwbfile = open_nwbfile(next(path for path in behavior_paths if get_modalities(path) == "image"))
print(behavior_nwbfile.session_description)
print()
print(f"Subject: {behavior_nwbfile.subject.subject_id}, {behavior_nwbfile.subject.genotype}")

In [ ]:
aim_df = behavior_nwbfile.events["aim_scores"].to_dataframe()
aim_df["minutes_after_levodopa"] = aim_df["timestamp"] / 60
aim_df.pivot(index="minutes_after_levodopa", columns="behavior", values="score")

In [ ]:
behavior_nwbfile.processing["behavior"]["AimEthogram"].to_dataframe()

In [ ]:
for name, video in behavior_nwbfile.acquisition.items():
    print(f"{name}: starts {video.starting_time / 60:.0f} min after levodopa, {video.external_file[0]}")

## Sr2+-oEPSC recordings

In these experiments the lab expressed channelrhodopsin in motor cortex and recorded SPNs in voltage clamp in a solution where strontium replaces calcium. A blue LED flash on the cortical terminals then evokes asynchronous excitatory postsynaptic currents, whose amplitude measures the strength of single corticostriatal synapses.

The files hold one mouse and recording day each, like the other patch-clamp files:

- One `VoltageClampSeries` per cell with its sweeps one after another, each at its own start time, and the intracellular recordings table pointing to each sweep's slice.
- `nwbfile.intervals["optogenetic_pulses_table"]`: the time of each LED flash, the cell recorded during it and the LED setting.
- The light source, the virus and its injection site in `nwbfile.lab_meta_data["optogenetic_experiment_metadata"]`.

In [ ]:
oepsc_path = (
    "sub-Subject20230705LIDONtDTomato/"
    "sub-Subject20230705LIDONtDTomato_ses-dSPN++OnState++none++WT++20230705_icephys.nwb"
)
oepsc_nwbfile = open_nwbfile(oepsc_path)
print(list(oepsc_nwbfile.acquisition))
print()
print(oepsc_nwbfile.acquisition["VoltageClampSeriesCell1"].description)

In [ ]:
pulses_df = oepsc_nwbfile.intervals["optogenetic_pulses_table"].to_dataframe()
pulses_df[["start_time", "stop_time", "cell_id", "led_setting", "wavelength_in_nm"]].head()

### Sweeps with the flash and the analysis window

The paper counted the asynchronous events from 40 ms to 400 ms after each flash (Methods). We plot two sweeps of cell 1 with the flash and that window, aligned on the sweep start.

In [ ]:
oepsc_recordings_df = oepsc_nwbfile.intracellular_recordings.to_dataframe()
cell_1_df = oepsc_recordings_df[oepsc_recordings_df[("intracellular_recordings", "cell_id")] == 1]
cell_1_pulses = pulses_df[pulses_df["cell_id"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, sweep in zip(axes, (0, 6)):
    row = cell_1_df.iloc[sweep]
    response = row[("responses", "response")]
    sweep_start = row[("intracellular_recordings", "sweep_start_time")]
    time_ms = (response.timestamps - response.timestamps[0]) * 1e3
    current_pA = response.data * response.timeseries.conversion * 1e12

    # The flash of this sweep
    flash = cell_1_pulses[(cell_1_pulses["start_time"] >= sweep_start)].iloc[0]
    flash_start_ms = (flash["start_time"] - sweep_start) * 1e3
    flash_stop_ms = (flash["stop_time"] - sweep_start) * 1e3

    ax.plot(time_ms, current_pA, lw=0.6)
    ax.axvspan(flash_start_ms, flash_stop_ms, color="cyan", label="LED flash")
    ax.axvspan(flash_start_ms + 40, flash_start_ms + 400, color="gray", alpha=0.15, label="40 to 400 ms after the flash")
    ax.set_xlim(0, 500)
    ax.set_title(f"Cell 1, sweep {sweep + 1}")
    ax.set_xlabel("Time from sweep start (ms)")
axes[0].set_ylabel("Current (pA)")
axes[0].legend()
plt.tight_layout()
plt.show()

### Event detection

As an example of analysis, we detect inward events in the 40 to 400 ms window. A running median over 20 ms follows the slow decay of the synchronous current, and an event is a drop below it of more than five robust standard deviations, estimated with the median absolute deviation (MAD). Crossings within 5 ms are merged into one event. This simple detector is for illustration; the paper's Methods describe the lab's analysis.

In [ ]:
from scipy.ndimage import median_filter


def merge_nearby_events(event_times, event_amplitudes, merge_distance_ms):
    """Merge events within merge_distance_ms of each other, keeping the largest amplitude."""
    order = np.argsort(event_times)
    times = np.asarray(event_times)[order]
    amplitudes = np.asarray(event_amplitudes)[order]
    merged_times, merged_amplitudes = [], []
    start = 0
    while start < len(times):
        stop = start + 1
        while stop < len(times) and times[stop] - times[start] <= merge_distance_ms:
            stop += 1
        largest = start + int(np.argmax(amplitudes[start:stop]))
        merged_times.append(times[largest])
        merged_amplitudes.append(amplitudes[largest])
        start = stop
    return np.array(merged_times), np.array(merged_amplitudes)


def detect_inward_events(row, pulses_of_cell, window_ms=(40, 400), threshold_mad=5, merge_distance_ms=5.0):
    """Inward events of one sweep in the window after its flash: times (ms after the flash) and amplitudes (pA)."""
    response = row[("responses", "response")]
    sweep_start = row[("intracellular_recordings", "sweep_start_time")]
    flash = pulses_of_cell[pulses_of_cell["start_time"] >= sweep_start].iloc[0]
    rate = round(1 / (response.timestamps[1] - response.timestamps[0]))
    time_ms = np.arange(response.count) / rate * 1e3 - (flash["start_time"] - sweep_start) * 1e3
    current_pA = response.data * response.timeseries.conversion * 1e12

    # Deviation from a 20 ms running median, which follows the slow decay of the synchronous current
    deviation = current_pA - median_filter(current_pA, size=int(0.02 * rate))
    in_window = (time_ms >= window_ms[0]) & (time_ms <= window_ms[1])
    robust_std = 1.4826 * np.median(np.abs(deviation[in_window] - np.median(deviation[in_window])))
    crossings = in_window & (deviation < -threshold_mad * robust_std)
    return merge_nearby_events(time_ms[crossings], -deviation[crossings], merge_distance_ms)


event_times, event_amplitudes = detect_inward_events(cell_1_df.iloc[0], cell_1_pulses)
print(f"Sweep 1 of cell 1: {len(event_times)} events, median amplitude {np.median(event_amplitudes):.1f} pA")

In [ ]:
# Every sweep of cell 1
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True, sharey=True)
all_amplitudes = []
for ax, (_, row) in zip(axes.ravel(), cell_1_df.iterrows()):
    response = row[("responses", "response")]
    sweep_start = row[("intracellular_recordings", "sweep_start_time")]
    flash = cell_1_pulses[cell_1_pulses["start_time"] >= sweep_start].iloc[0]
    time_ms = (response.timestamps - flash["start_time"]) * 1e3
    event_times, event_amplitudes = detect_inward_events(row, cell_1_pulses)
    all_amplitudes.extend(event_amplitudes)

    current_pA = response.data * response.timeseries.conversion * 1e12
    ax.plot(time_ms, current_pA, lw=0.5)
    ax.plot(event_times, np.interp(event_times, time_ms, current_pA), "r.", markersize=5)
    ax.axvspan(40, 400, color="gray", alpha=0.15)
    ax.set_xlim(-20, 450)
    ax.set_title(f"Sweep at {sweep_start:.0f} s: {len(event_times)} events", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("Time from the flash (ms)")
for ax in axes[:, 0]:
    ax.set_ylabel("Current (pA)")
plt.tight_layout()
plt.show()

print(f"Cell 1: {len(all_amplitudes)} events, median amplitude {np.median(all_amplitudes):.1f} pA")

## Acetylcholine release: GRAB-ACh3.0 imaging

The lab imaged the acetylcholine sensor GRAB-ACh3.0 in slices of the dorsolateral striatum while stimulating locally with an electrode, in unlesioned mice, 6-OHDA-lesioned mice and mice in the LID off-state. Each region was filmed in a series of trials while the drug in the bath changed, and was calibrated with acetylcholine (maximum fluorescence) and tetrodotoxin (minimum).

The files hold one mouse and recording day each:

- One `TwoPhotonSeries` per region and channel with the region's trials back to back: GRAB-ACh3.0 fluorescence, and the Dodt contrast image of the tissue.
- One `RoiResponseSeries` per region and channel in `processing/ophys/Fluorescence`: the brightness of the whole frame that Prairie View computed during recording.
- The trials table: one row per trial, with the region, the drug, the stimulation, the stimulus time, and each trial's slices of the series.

In [ ]:
grab_ach_paths = sorted(asset_path for asset_path in assets if get_modalities(asset_path) == "ophys")
for asset_path in grab_ach_paths:
    print(get_session_id(asset_path))

In [ ]:
grab_ach_nwbfile = open_nwbfile(grab_ach_paths[0])
trials_df = grab_ach_nwbfile.trials.to_dataframe()
trials_df[["start_time", "stop_time", "region", "drug", "stimulation", "stimulus_start_time"]].head(10)

In [ ]:
trials_df.groupby(["region", "drug", "stimulation"]).size().unstack(fill_value=0)

### Fluorescence of a region's trials

Each trial's row points to its slice of the region's fluorescence in the `roi_response_grabach3` column. We plot the single-pulse trials of one region in each drug, aligned on the stimulus.

In [ ]:
region = trials_df["region"].iloc[0]
single_pulse_df = trials_df[(trials_df["region"] == region) & (trials_df["stimulation"] == "single 1 ms pulse")]

fig, axes = plt.subplots(1, single_pulse_df["drug"].nunique(), figsize=(15, 4), sharey=True)
for ax, (drug, drug_df) in zip(axes, single_pulse_df.groupby("drug")):
    for _, trial in drug_df.iterrows():
        fluorescence = trial["roi_response_grabach3"]
        ax.plot(fluorescence.timestamps - trial["stimulus_start_time"], fluorescence.data, lw=1)
    ax.axvline(0, color="r", linestyle="--", lw=1)
    ax.set_title(f"{region}, {drug}")
    ax.set_xlabel("Time from stimulus (s)")
axes[0].set_ylabel("GRAB-ACh3.0 fluorescence (a.u.)")
plt.tight_layout()
plt.show()

The movie frames of a trial are in its `two_photon_grabach3` slice. Here is the mean frame of the first trial.

In [ ]:
movie = trials_df.iloc[0]["two_photon_grabach3"]
mean_frame = np.asarray(movie.data).mean(axis=0)
plt.imshow(mean_frame, cmap="gray")
plt.title(f"{movie.timeseries.name}, mean of the first trial's {movie.count} frames")
plt.axis("off")
plt.show()